# 🛒 RetailSense — Demand Forecasting & Inventory Intelligence

**Dataset:** Rossmann Store Sales (Kaggle public)

**Problem:** Forecast daily sales 42 days ahead per store to minimize inventory cost (stockouts + overstock).

**Pipeline:**
1. Data loading & cleaning
2. Feature engineering (lag, rolling, Fourier, calendar)
3. Model comparison: XGBoost · LightGBM · Prophet · Stacking Ensemble
4. Walk-forward cross-validation (time-safe)
5. Quantile regression (p10/p50/p90) for uncertainty
6. Hyperparameter tuning with Optuna
7. SHAP explainability
8. Business cost optimization
9. Artifact export for Streamlit

**Runtime:** ~12 minutes on Colab Free Tier · RAM < 4GB · No GPU needed

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install lightgbm xgboost optuna prophet shap plotly kaggle --quiet
print('✅ All packages installed')

## 📦 Cell 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pickle, json, zipfile, os, time
from pathlib import Path
from datetime import datetime, timedelta

# ML
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# SHAP
import shap
shap.initjs()

# Prophet
from prophet import Prophet

# Plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Artifacts dir
Path('/content/artifacts').mkdir(exist_ok=True)

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

start_time = time.time()
print('✅ Imports complete')

## 📥 Cell 3 — Download Dataset from Kaggle

> **Setup:** Upload your `kaggle.json` API key, or use the manual download link below.
>
> **Manual option:** Download from https://www.kaggle.com/c/rossmann-store-sales/data and upload `train.csv` and `store.csv` to `/content/`

In [ ]:
# ── Option A: Kaggle API (recommended) ───────────────────────────────────
import os
from google.colab import files

USE_KAGGLE_API = True   # Set False if uploading CSVs manually

if USE_KAGGLE_API:
    print('📤 Upload your kaggle.json ...')
    uploaded = files.upload()   # upload kaggle.json
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    os.system('cp kaggle.json /root/.config/kaggle/kaggle.json')
    os.system('chmod 600 /root/.config/kaggle/kaggle.json')
    os.system('kaggle competitions download -c rossmann-store-sales -p /content/')
    os.system('cd /content && unzip -o rossmann-store-sales.zip')
    print('✅ Dataset downloaded')
else:
    print('📤 Please upload train.csv and store.csv manually via Files panel')
    # files.upload()   # uncomment to upload manually

## 🧹 Cell 4 — Load & Clean Data

In [ ]:
t0 = time.time()

# ── Load ─────────────────────────────────────────────────────────────────
train = pd.read_csv('/content/train.csv', parse_dates=['Date'], low_memory=False)
store = pd.read_csv('/content/store.csv')

print(f'Train shape : {train.shape}')
print(f'Store shape : {store.shape}')
print(f'Date range  : {train.Date.min()} → {train.Date.max()}')

# ── Merge ─────────────────────────────────────────────────────────────────
df = train.merge(store, on='Store', how='left')

# ── Drop closed stores (Sales=0 when Open=0 is not a real demand signal) ──
df = df[df['Open'] == 1].copy()
df = df[df['Sales'] > 0].copy()

# ── Fix missing values ────────────────────────────────────────────────────
df['CompetitionDistance'].fillna(df['CompetitionDistance'].median(), inplace=True)
df['CompetitionOpenSinceMonth'].fillna(0, inplace=True)
df['CompetitionOpenSinceYear'].fillna(0, inplace=True)
df['Promo2SinceWeek'].fillna(0, inplace=True)
df['Promo2SinceYear'].fillna(0, inplace=True)
df['PromoInterval'].fillna('None', inplace=True)

# ── Encode categoricals ───────────────────────────────────────────────────
le = LabelEncoder()
df['StoreType']     = le.fit_transform(df['StoreType'].astype(str))
df['Assortment']    = le.fit_transform(df['Assortment'].astype(str))
df['StateHoliday']  = le.fit_transform(df['StateHoliday'].astype(str))
df['PromoInterval'] = le.fit_transform(df['PromoInterval'].astype(str))

# ── Sort chronologically ──────────────────────────────────────────────────
df.sort_values(['Store', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nCleaned shape : {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'⏱  Load+clean  : {time.time()-t0:.1f}s')
df.head(3)

## 📊 Cell 5 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RetailSense — Exploratory Analysis', fontsize=16, fontweight='bold')

# 1. Sales distribution
axes[0,0].hist(df['Sales'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,0].set_title('Sales Distribution')
axes[0,0].set_xlabel('Daily Sales (€)')

# 2. Mean sales by day of week
dow = df.groupby('DayOfWeek')['Sales'].mean()
dow.plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='white')
axes[0,1].set_title('Avg Sales by Day of Week')
axes[0,1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)

# 3. Promo effect
promo_effect = df.groupby('Promo')['Sales'].mean()
promo_effect.plot(kind='bar', ax=axes[0,2], color=['tomato','mediumseagreen'], edgecolor='white')
axes[0,2].set_title('Promotion Lift')
axes[0,2].set_xticklabels(['No Promo', 'Promo'], rotation=0)

# 4. Monthly aggregated trend (sample 5 stores)
sample_stores = [1, 2, 3, 85, 262]
for s in sample_stores:
    subset = df[df['Store'] == s].set_index('Date')['Sales'].resample('W').mean()
    axes[1,0].plot(subset.index, subset.values, alpha=0.7, lw=1.2)
axes[1,0].set_title('Weekly Sales — Sample Stores')
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[1,0].xaxis.get_majorticklabels(), rotation=30)

# 5. Competition distance vs median sales
df['CompDist_bin'] = pd.cut(df['CompetitionDistance'], bins=5)
comp_sales = df.groupby('CompDist_bin')['Sales'].median()
axes[1,1].bar(range(len(comp_sales)), comp_sales.values, color='mediumpurple', edgecolor='white')
axes[1,1].set_title('Competition Distance vs Median Sales')
axes[1,1].set_xticklabels([])
axes[1,1].set_xlabel('Competition Distance (binned far →)')

# 6. Store type distribution
store_type_sales = df.groupby('StoreType')['Sales'].mean()
axes[1,2].bar(range(len(store_type_sales)), store_type_sales.values,
              color=['#2196F3','#FF5722','#4CAF50','#FF9800'], edgecolor='white')
axes[1,2].set_title('Avg Sales by Store Type')
axes[1,2].set_xticks(range(len(store_type_sales)))
axes[1,2].set_xticklabels([f'Type {i}' for i in range(len(store_type_sales))])

plt.tight_layout()
plt.savefig('/content/artifacts/eda.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved')

## 🔧 Cell 6 — Feature Engineering

Key insight: we frame time series forecasting as a **supervised ML problem** by generating lag and rolling features. This is what separates ML-based forecasting from pure statistical methods.

In [ ]:
t0 = time.time()

def add_calendar_features(df):
    """Extract rich temporal features from date."""
    df = df.copy()
    df['Year']        = df['Date'].dt.year
    df['Month']       = df['Date'].dt.month
    df['Day']         = df['Date'].dt.day
    df['WeekOfYear']  = df['Date'].dt.isocalendar().week.astype(int)
    df['Quarter']     = df['Date'].dt.quarter
    df['IsWeekend']   = (df['DayOfWeek'] >= 6).astype(int)
    df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
    df['IsMonthEnd']  = df['Date'].dt.is_month_end.astype(int)

    # Fourier terms: capture weekly and annual seasonality
    day_of_year = df['Date'].dt.dayofyear
    df['sin_week']  = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
    df['cos_week']  = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
    df['sin_year']  = np.sin(2 * np.pi * day_of_year / 365)
    df['cos_year']  = np.cos(2 * np.pi * day_of_year / 365)
    df['sin_month'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['cos_month'] = np.cos(2 * np.pi * df['Month'] / 12)
    return df


def add_lag_features(df, lags=[7, 14, 28, 56]):
    """Lag features: past sales at meaningful retail horizons."""
    df = df.copy()
    for lag in lags:
        df[f'Sales_lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)
    return df


def add_rolling_features(df, windows=[7, 14, 28]):
    """Rolling statistics: local trend + volatility signals."""
    df = df.copy()
    for w in windows:
        rolled = df.groupby('Store')['Sales'].shift(1).rolling(w)
        df[f'Sales_roll_mean_{w}']  = rolled.mean().reset_index(level=0, drop=True)
        df[f'Sales_roll_std_{w}']   = rolled.std().reset_index(level=0, drop=True)
        df[f'Sales_roll_max_{w}']   = rolled.max().reset_index(level=0, drop=True)

    # Expanding mean (store-level average — very strong signal)
    df['Sales_expanding_mean'] = (
        df.groupby('Store')['Sales']
          .transform(lambda x: x.shift(1).expanding().mean())
    )
    return df


def add_competition_age(df):
    """How long has a competitor been open? (months)"""
    df = df.copy()
    df['CompetitionAge'] = (
        12 * (df['Year'] - df['CompetitionOpenSinceYear']) +
        (df['Month'] - df['CompetitionOpenSinceMonth'])
    ).clip(lower=0)
    return df


def add_promo2_features(df):
    """Was Promo2 active this month?"""
    df = df.copy()
    month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
                 'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
    # PromoInterval already label-encoded; approximate with binary flag
    df['Promo2Active'] = (
        (df['Promo2'] == 1) &
        (df['Year'] >= df['Promo2SinceYear'])
    ).astype(int)
    return df


# ── Apply all feature engineering ────────────────────────────────────────
df = add_calendar_features(df)
df = add_lag_features(df)
df = add_rolling_features(df)
df = add_competition_age(df)
df = add_promo2_features(df)

# ── Drop rows with NaN lags (first 56 days per store) ────────────────────
df.dropna(subset=[c for c in df.columns if 'lag' in c or 'roll' in c], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Feature-engineered shape: {df.shape}')
print(f'New features count: {df.shape[1]}')
print(f'⏱  Feature engineering: {time.time()-t0:.1f}s')

# Save engineered features list for Streamlit
feature_cols = [
    'Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday',
    'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionAge',
    'Promo2', 'Promo2Active', 'Year', 'Month', 'Day', 'WeekOfYear', 'Quarter',
    'IsWeekend', 'IsMonthStart', 'IsMonthEnd',
    'sin_week', 'cos_week', 'sin_year', 'cos_year', 'sin_month', 'cos_month',
    'Sales_lag_7', 'Sales_lag_14', 'Sales_lag_28', 'Sales_lag_56',
    'Sales_roll_mean_7', 'Sales_roll_std_7', 'Sales_roll_max_7',
    'Sales_roll_mean_14', 'Sales_roll_std_14', 'Sales_roll_max_14',
    'Sales_roll_mean_28', 'Sales_roll_std_28', 'Sales_roll_max_28',
    'Sales_expanding_mean'
]

TARGET = 'Sales'
print(f'\nFeatures used: {len(feature_cols)}')

## ✂️ Cell 7 — Walk-Forward Train/Validation Split

**Critical:** Never use random splits for time series. Using future data to predict the past is **data leakage** and produces inflated metrics that don't hold in production. We use walk-forward validation: train on all data before a cutoff, validate on the next 6 weeks.

In [ ]:
# Last 6 weeks = validation set (42 days forecast horizon)
CUTOFF_DATE = df['Date'].max() - timedelta(days=42)

train_df = df[df['Date'] <= CUTOFF_DATE].copy()
val_df   = df[df['Date'] >  CUTOFF_DATE].copy()

X_train = train_df[feature_cols]
y_train = np.log1p(train_df[TARGET])   # log-transform: normalizes heavy-tailed sales

X_val   = val_df[feature_cols]
y_val   = np.log1p(val_df[TARGET])

# True scale (not log) for business metrics
y_val_true = val_df[TARGET].values

print(f'Training period : {train_df.Date.min().date()} → {train_df.Date.max().date()}')
print(f'Validation period: {val_df.Date.min().date()} → {val_df.Date.max().date()}')
print(f'Train rows: {len(X_train):,} | Val rows: {len(X_val):,}')


def rmspe(y_true, y_pred):
    """Root Mean Squared Percentage Error (Kaggle Rossmann metric)."""
    mask = y_true > 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

def evaluate(y_true, y_pred_log, label='Model'):
    y_pred = np.expm1(y_pred_log)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    mape  = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmspe_val = rmspe(y_true, y_pred)
    print(f'{label:30s} | RMSPE: {rmspe_val:.4f} | MAE: {mae:,.0f} | MAPE: {mape:.1f}% | RMSE: {rmse:,.0f}')
    return {'model': label, 'RMSPE': rmspe_val, 'MAE': mae, 'MAPE': mape, 'RMSE': rmse}

results = []
print('\n✅ Walk-forward split ready')

## 🤖 Cell 8 — Model 1: XGBoost Baseline

In [ ]:
t0 = time.time()

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
    tree_method='hist'
)

xgb_model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)

xgb_preds = xgb_model.predict(X_val)
r = evaluate(y_val_true, xgb_preds, 'XGBoost (baseline)')
results.append(r)
print(f'⏱  XGBoost: {time.time()-t0:.1f}s')

## 🤖 Cell 9 — Model 2: LightGBM Baseline

In [ ]:
t0 = time.time()

lgb_model = lgb.LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])

lgb_preds = lgb_model.predict(X_val)
r = evaluate(y_val_true, lgb_preds, 'LightGBM (baseline)')
results.append(r)
print(f'⏱  LightGBM: {time.time()-t0:.1f}s')

## 🤖 Cell 10 — Model 3: Prophet (sample of stores)

Prophet excels at capturing holidays and changepoints. We train on a representative subset for speed.

In [ ]:
t0 = time.time()

# Train Prophet on top-10 stores by volume (for speed)
top_stores = df.groupby('Store')['Sales'].sum().nlargest(10).index.tolist()

prophet_val_preds = []
prophet_val_true  = []

for store_id in top_stores:
    store_df = df[df['Store'] == store_id][['Date', 'Sales']].copy()
    store_df.columns = ['ds', 'y']
    store_df['y'] = np.log1p(store_df['y'])

    tr = store_df[store_df['ds'] <= CUTOFF_DATE]
    va = store_df[store_df['ds'] >  CUTOFF_DATE]

    if len(tr) < 100 or len(va) == 0:
        continue

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05
    )
    m.fit(tr)

    future = m.make_future_dataframe(periods=42)
    forecast = m.predict(future)
    preds_log = forecast.tail(len(va))['yhat'].values

    prophet_val_preds.extend(preds_log)
    prophet_val_true.extend(np.expm1(va['y'].values))

prophet_val_preds = np.array(prophet_val_preds)
prophet_val_true  = np.array(prophet_val_true)

# Clip negatives before expm1
prophet_val_preds = np.clip(prophet_val_preds, 0, None)

prophet_rmspe = rmspe(prophet_val_true, np.expm1(prophet_val_preds))
prophet_mae   = mean_absolute_error(prophet_val_true, np.expm1(prophet_val_preds))
print(f'Prophet (top-10 stores)           | RMSPE: {prophet_rmspe:.4f} | MAE: {prophet_mae:,.0f}')
results.append({'model': 'Prophet (top-10 stores)', 'RMSPE': prophet_rmspe,
                'MAE': prophet_mae, 'MAPE': 0, 'RMSE': 0})
print(f'⏱  Prophet: {time.time()-t0:.1f}s')

## ⚡ Cell 11 — Optuna Hyperparameter Tuning (LightGBM)

In [ ]:
t0 = time.time()

# Sub-sample for faster tuning (representative 20%)
sample_idx = np.random.choice(len(X_train), size=int(0.2 * len(X_train)), replace=False)
X_tune = X_train.iloc[sample_idx]
y_tune = y_train.iloc[sample_idx]

def lgb_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 200, 600),
        'learning_rate':   trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
        'num_leaves':      trial.suggest_int('num_leaves', 31, 127),
        'max_depth':       trial.suggest_int('max_depth', 4, 8),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':      trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }
    model = lgb.LGBMRegressor(**params)
    # Time-series-safe: 3-fold with gap
    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for tr_idx, va_idx in tscv.split(X_tune):
        model.fit(X_tune.iloc[tr_idx], y_tune.iloc[tr_idx])
        preds = model.predict(X_tune.iloc[va_idx])
        # RMSPE in log space ≈ MAPE
        true = np.expm1(y_tune.iloc[va_idx].values)
        pred = np.expm1(preds)
        score = rmspe(true, pred)
        scores.append(score)
    return np.mean(scores)

study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(lgb_objective, n_trials=25, timeout=180)   # 3-min cap

print(f'\nBest RMSPE (CV): {study.best_value:.4f}')
print(f'Best params: {study.best_params}')
print(f'⏱  Optuna: {time.time()-t0:.1f}s')

## 🏆 Cell 12 — Final Tuned LightGBM Model

In [ ]:
t0 = time.time()

best_params = study.best_params
best_params.update({'n_jobs': -1, 'random_state': 42, 'verbose': -1})

lgb_tuned = lgb.LGBMRegressor(**best_params)
lgb_tuned.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])

lgb_tuned_preds = lgb_tuned.predict(X_val)
r = evaluate(y_val_true, lgb_tuned_preds, 'LightGBM (Optuna tuned)')
results.append(r)
print(f'⏱  Tuned LightGBM: {time.time()-t0:.1f}s')

## 🔗 Cell 13 — Stacking Ensemble

In [ ]:
t0 = time.time()

# Stack XGBoost + LightGBM tuned predictions with Ridge meta-learner
stack_train_preds = np.column_stack([
    xgb_model.predict(X_train),
    lgb_tuned.predict(X_train)
])

stack_val_preds = np.column_stack([
    xgb_preds,
    lgb_tuned_preds
])

meta_model = Ridge(alpha=1.0)
meta_model.fit(stack_train_preds, y_train)

ensemble_preds = meta_model.predict(stack_val_preds)
r = evaluate(y_val_true, ensemble_preds, 'Stacking Ensemble (final)')
results.append(r)

print(f'\nMeta-learner weights: XGB={meta_model.coef_[0]:.3f}, LGB={meta_model.coef_[1]:.3f}')
print(f'⏱  Ensemble: {time.time()-t0:.1f}s')

## 📊 Cell 14 — Model Comparison Plot

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('RMSPE')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')

colors = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0']

# RMSPE
axes[0].barh(results_df['model'], results_df['RMSPE'],
             color=colors[:len(results_df)], edgecolor='white')
axes[0].set_title('RMSPE (lower = better)')
axes[0].set_xlabel('RMSPE')
for i, v in enumerate(results_df['RMSPE']):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

# MAE
axes[1].barh(results_df['model'], results_df['MAE'],
             color=colors[:len(results_df)], edgecolor='white')
axes[1].set_title('MAE — Mean Absolute Error (€)')
axes[1].set_xlabel('MAE (€)')
for i, v in enumerate(results_df['MAE']):
    axes[1].text(v + 10, i, f'€{v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/artifacts/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n📊 Model Leaderboard:')
print(results_df[['model','RMSPE','MAE']].to_string(index=False))

## 📐 Cell 15 — Quantile Regression (Uncertainty Intervals)

Point predictions aren't enough for inventory decisions. A forecast of 5,000 units with ±500 variance requires a completely different order quantity than 5,000 ±2,000. We train three models: p10, p50, p90.

In [ ]:
t0 = time.time()

quantile_models = {}

for q, alpha in [(0.1, 'p10'), (0.5, 'p50'), (0.9, 'p90')]:
    qmodel = lgb.LGBMRegressor(
        objective='quantile',
        alpha=q,
        n_estimators=best_params.get('n_estimators', 300),
        learning_rate=best_params.get('learning_rate', 0.05),
        num_leaves=best_params.get('num_leaves', 63),
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )
    qmodel.fit(X_train, y_train)
    quantile_models[alpha] = qmodel
    print(f'  ✅ Quantile model {alpha} (q={q}) trained')

# Coverage probability on validation set
p10_pred = np.expm1(quantile_models['p10'].predict(X_val))
p90_pred = np.expm1(quantile_models['p90'].predict(X_val))
coverage = np.mean((y_val_true >= p10_pred) & (y_val_true <= p90_pred))
print(f'\nPI Coverage (p10–p90): {coverage:.1%} (target ~80%)')
print(f'⏱  Quantile models: {time.time()-t0:.1f}s')

## 📈 Cell 16 — Forecast Visualization (Single Store)

In [ ]:
DEMO_STORE = 1   # Change to any store 1-1115

store_val = val_df[val_df['Store'] == DEMO_STORE].copy()
store_X   = store_val[feature_cols]

store_p50 = np.expm1(quantile_models['p50'].predict(store_X))
store_p10 = np.expm1(quantile_models['p10'].predict(store_X))
store_p90 = np.expm1(quantile_models['p90'].predict(store_X))

# Also pull 60-day history for context
store_hist = train_df[train_df['Store'] == DEMO_STORE].tail(60)

fig = go.Figure()

# Historical
fig.add_trace(go.Scatter(
    x=store_hist['Date'], y=store_hist['Sales'],
    name='Historical Sales', line=dict(color='#607D8B', width=1.5)
))

# PI band
fig.add_trace(go.Scatter(
    x=pd.concat([store_val['Date'], store_val['Date'][::-1]]),
    y=np.concatenate([store_p90, store_p10[::-1]]),
    fill='toself', fillcolor='rgba(33,150,243,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='80% Prediction Interval'
))

# Actual
fig.add_trace(go.Scatter(
    x=store_val['Date'], y=store_val['Sales'],
    name='Actual Sales', line=dict(color='#4CAF50', width=1.5)
))

# Forecast
fig.add_trace(go.Scatter(
    x=store_val['Date'], y=store_p50,
    name='Forecast (p50)', line=dict(color='#2196F3', width=2.5, dash='dot')
))

fig.update_layout(
    title=f'Store {DEMO_STORE} — 42-Day Forecast with Uncertainty Intervals',
    xaxis_title='Date', yaxis_title='Daily Sales (€)',
    height=420, template='plotly_white',
    legend=dict(orientation='h', y=-0.2)
)

fig.write_html('/content/artifacts/forecast_plot.html')
fig.show()
print('✅ Forecast plot saved')

## 🔍 Cell 17 — SHAP Explainability

In [ ]:
t0 = time.time()

# SHAP on a 2000-row sample for speed
shap_sample = X_val.sample(2000, random_state=42)

explainer   = shap.TreeExplainer(lgb_tuned)
shap_values = explainer.shap_values(shap_sample)

# Global feature importance
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plt.sca(axes[0])
shap.summary_plot(shap_values, shap_sample, plot_type='bar',
                  max_display=20, show=False)
axes[0].set_title('SHAP Feature Importance (Global)', fontsize=12)

plt.sca(axes[1])
shap.summary_plot(shap_values, shap_sample, max_display=20, show=False)
axes[1].set_title('SHAP Beeswarm — Feature Impact', fontsize=12)

plt.tight_layout()
plt.savefig('/content/artifacts/shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

# Single prediction waterfall (for Streamlit explainability tab)
single_idx = 0
fig2, ax2 = plt.subplots(figsize=(10, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[single_idx],
        base_values=explainer.expected_value,
        data=shap_sample.iloc[single_idx],
        feature_names=feature_cols
    ), show=False
)
plt.title('SHAP Waterfall — Single Prediction Explanation', fontsize=11)
plt.tight_layout()
plt.savefig('/content/artifacts/shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'⏱  SHAP: {time.time()-t0:.1f}s')

# Top features
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_cols
).sort_values(ascending=False)
print('\nTop 10 features by SHAP importance:')
print(mean_shap.head(10).to_string())

## 💰 Cell 18 — Business Cost Optimization

The forecast is only valuable if it improves real business decisions. We use the prediction intervals to compute the **optimal safety stock** that minimizes total inventory cost.

In [ ]:
# ── Cost parameters (realistic German retail) ─────────────────────────────
UNIT_COST         = 8.0    # € cost to produce/buy one unit
SELLING_PRICE     = 15.0   # € revenue per unit sold
HOLDING_COST_RATE = 0.20   # 20% of unit cost per year = 0.055% per day
STOCKOUT_PENALTY  = 5.0    # € lost goodwill per unit unavailable

holding_cost_day  = UNIT_COST * HOLDING_COST_RATE / 365
stockout_cost     = SELLING_PRICE - UNIT_COST + STOCKOUT_PENALTY   # lost margin + penalty

print(f'Cost setup:')
print(f'  Holding cost/unit/day : €{holding_cost_day:.4f}')
print(f'  Stockout cost/unit    : €{stockout_cost:.2f}')
print(f'  Critical ratio        : {stockout_cost/(stockout_cost+holding_cost_day):.3f}')

# ── Critical Ratio → Optimal Quantile ─────────────────────────────────────
# From newsvendor model: order at quantile q* = Co / (Co + Cu)
# Co = overage cost (holding), Cu = underage cost (stockout)
critical_ratio = stockout_cost / (stockout_cost + holding_cost_day)
print(f'\nOptimal service level (newsvendor): {critical_ratio:.1%}')
print('→ Order at the ~98th percentile of demand forecast')

# ── Compare naive vs intelligent ordering on validation set ───────────────
sample_store_ids = val_df['Store'].unique()[:50]  # 50 stores
sample_val = val_df[val_df['Store'].isin(sample_store_ids)]
sample_X   = sample_val[feature_cols]

# Naive: order yesterday's sales (common baseline)
naive_order    = np.expm1(quantile_models['p50'].predict(sample_X)) * 0.95  # slightly under
# Intelligent: order at p80 (between p50 and p90)
p80_model = lgb.LGBMRegressor(
    objective='quantile', alpha=0.80,
    n_estimators=best_params.get('n_estimators', 300),
    learning_rate=best_params.get('learning_rate', 0.05),
    num_leaves=best_params.get('num_leaves', 63),
    n_jobs=-1, random_state=42, verbose=-1
)
p80_model.fit(X_train, y_train)
intelligent_order = np.expm1(p80_model.predict(sample_X))
actual_demand = sample_val['Sales'].values

def compute_cost(order, demand, h=holding_cost_day, s=stockout_cost):
    overstock  = np.maximum(order - demand, 0)
    understock = np.maximum(demand - order, 0)
    return np.sum(overstock * h + understock * s)

naive_cost_total = compute_cost(naive_order, actual_demand)
intel_cost_total = compute_cost(intelligent_order, actual_demand)
savings_pct = (naive_cost_total - intel_cost_total) / naive_cost_total * 100

print(f'\n{'─'*50}')
print(f'Naive ordering cost     : €{naive_cost_total:>12,.0f}')
print(f'Intelligent ordering cost: €{intel_cost_total:>12,.0f}')
print(f'Cost reduction          :  {savings_pct:.1f}%')
print(f'{'─'*50}')

# Save cost params for Streamlit
cost_params = {
    'unit_cost': UNIT_COST,
    'selling_price': SELLING_PRICE,
    'holding_cost_rate': HOLDING_COST_RATE,
    'stockout_penalty': STOCKOUT_PENALTY,
    'critical_ratio': critical_ratio,
    'savings_pct': savings_pct
}
with open('/content/artifacts/cost_params.json', 'w') as f:
    json.dump(cost_params, f, indent=2)
print('✅ Cost params saved')

## 💾 Cell 19 — Export Artifacts for Streamlit

In [ ]:
import pickle, json

ART = Path('/content/artifacts')

# 1. Models
with open(ART / 'lgbm_tuned.pkl', 'wb') as f:
    pickle.dump(lgb_tuned, f)

with open(ART / 'xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

with open(ART / 'meta_model.pkl', 'wb') as f:
    pickle.dump(meta_model, f)

# 2. Quantile models
with open(ART / 'quantile_models.pkl', 'wb') as f:
    pickle.dump(quantile_models, f)

with open(ART / 'p80_model.pkl', 'wb') as f:
    pickle.dump(p80_model, f)

# 3. SHAP explainer
with open(ART / 'shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)

# 4. Metadata
best_result = results_df.iloc[0]
metadata = {
    'best_model': best_result['model'],
    'rmspe': float(best_result['RMSPE']),
    'mae':   float(best_result['MAE']),
    'feature_cols': feature_cols,
    'cutoff_date': str(CUTOFF_DATE.date()),
    'val_start': str(val_df.Date.min().date()),
    'val_end': str(val_df.Date.max().date()),
    'n_stores': int(df['Store'].nunique()),
    'pi_coverage': float(coverage),
    'best_params': study.best_params,
    'model_results': results_df[['model','RMSPE','MAE']].to_dict('records')
}
with open(ART / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# 5. Pre-compute forecasts for all stores (for fast Streamlit loading)
print('Pre-computing store forecasts...')
store_forecasts = {}
for store_id in val_df['Store'].unique()[:200]:   # first 200 stores
    sv = val_df[val_df['Store'] == store_id]
    if len(sv) < 5:
        continue
    sx = sv[feature_cols]
    store_forecasts[int(store_id)] = {
        'dates': sv['Date'].dt.strftime('%Y-%m-%d').tolist(),
        'actual': sv['Sales'].tolist(),
        'p10': np.expm1(quantile_models['p10'].predict(sx)).tolist(),
        'p50': np.expm1(quantile_models['p50'].predict(sx)).tolist(),
        'p90': np.expm1(quantile_models['p90'].predict(sx)).tolist(),
    }

with open(ART / 'store_forecasts.json', 'w') as f:
    json.dump(store_forecasts, f)

# 6. Store metadata
store_meta = store[['Store','StoreType','Assortment','CompetitionDistance']].copy()
store_meta.to_csv(ART / 'store_meta.csv', index=False)

# 7. SHAP values (sample)
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df.to_parquet(ART / 'shap_values_sample.parquet', index=False)
shap_sample.to_parquet(ART / 'shap_X_sample.parquet', index=False)

print('\n✅ All artifacts saved:')
for f in sorted(ART.iterdir()):
    size = f.stat().st_size / 1024
    print(f'   {f.name:<40} {size:>8.1f} KB')

## 📦 Cell 20 — Final Summary & Download

In [ ]:
total_time = time.time() - start_time

print('═' * 65)
print('       RETAILSENSE — DEMAND FORECASTING — FINAL RESULTS')
print('═' * 65)
print(f'Dataset:            Rossmann Store Sales')
print(f'Stores:             {df["Store"].nunique():,}')
print(f'Training samples:   {len(X_train):,}')
print(f'Features engineered:{len(feature_cols)}')
print(f'Forecast horizon:   42 days')
print()
print('── Model Leaderboard ──')
print(f'{"Model":<32} {"RMSPE":>8} {"MAE (€)":>10}')
print('─' * 55)
for _, row in results_df.iterrows():
    marker = '★' if row['model'] == results_df.iloc[0]['model'] else ' '
    print(f'{marker} {row["model"]:<30} {row["RMSPE"]:>8.4f} {row["MAE"]:>10,.0f}')
print()
print('── Business Impact ──')
print(f'PI Coverage (p10-p90):   {coverage:.1%}')
print(f'Inventory cost reduction: {savings_pct:.1f}% vs naive ordering')
print()
print(f'Total runtime: {total_time/60:.1f} minutes')
print('═' * 65)

# Zip and download
!cd /content && zip -r retailsense_artifacts.zip artifacts/ -q
from google.colab import files
files.download('/content/retailsense_artifacts.zip')
print('\n📥 Download started — extract to streamlit_app/artifacts/')